In [1]:
from rsome import dro
from rsome import norm
from rsome import E
from rsome import eco_solver as eco
from rsome import grb_solver as grb
import numpy as np
import numpy.random as rd

# model and ambiguity set parameters
I = 2
S = 50
c = np.ones(I)
d = 50 * I
p = 1 + 4*rd.rand(I)
zbar = 100 * rd.rand(I)
zhat = zbar * rd.rand(S, I)
theta = 0.01 * zbar.min()

# modeling with RSOME
model = dro.Model(S)                        # create a DRO model with S scenarios
z = model.rvar(I)                           # random demand z
u = model.rvar()                            # auxiliary random variable

fset = model.ambiguity()                    # create an ambiguity set
for s in range(S):
    fset[s].suppset(0 <= z, z <= zbar,
                    norm(z - zhat[s]) <= u) # define the support for each scenario
fset.exptset(E(u) <= theta)                 # the Wasserstein metric constraint
pr = model.p                                # an array of scenario probabilities
fset.probset(pr == 1/S)                     # support of scenario probabilities

x = model.dvar(I)                           # define first-stage decisions
y = model.dvar(I)                           # define decision rule variables
y.adapt(z)                                  # y affinely adapts to z
y.adapt(u)                                  # y affinely adapts to u
for s in range(S):
    y.adapt(s)                              # y adapts to each scenario s

model.minsup(-p@x + E(p@y), fset)           # worst-case expectation over fset
model.st(y >= 0)                            # constraints
model.st(y >= x - z)                        # constraints
model.st(x >= 0)                            # constraints
model.st(c@x == d)                          # constraints

model.solve(eco)  

Being solved by ECOS...
Solution status: Close to optimal solution found
Running time: 0.0321s


In [4]:
model.do_math().show()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x1802,x1803,x1804,x1805,x1806,x1807,x1808,x1809,sense,constant
Obj,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-,-
LC1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<=,-0.0
LC2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,-0.0
LC3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,-0.0
LC4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
QC249,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<=,0.0
QC250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,-1.0,0.0,0.0,<=,0.0
UB,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,0.0,0.0,inf,inf,inf,0.0,0.0,-,-
LB,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,0.0,-inf,-inf,-inf,-inf,0.0,-inf,-inf,-,-


In [5]:
model.do_math(primal=False).show()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x1998,x1999,x2000,x2001,x2002,x2003,x2004,x2005,sense,constant
Obj,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-,-
LC1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,1.0
LC2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,0.0
LC3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,0.0
LC4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,==,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
QC249,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.0,1.0,1.0,0.0,0.0,0.0,<=,0.0
QC250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-1.0,1.0,1.0,<=,0.0
UB,0.0,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,-,-
LB,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,0.0,-inf,-inf,0.0,-inf,-inf,-,-
